# CareerPath AI - Modeling Pipeline
Notebook ini berisi tahapan pemodelan inti untuk aplikasi CareerPath AI. Model ini berfungsi untuk mencocokkan *skill* yang dimiliki pengguna dengan kebutuhan industri menggunakan metode **TF-IDF** dan **Cosine Similarity**, lalu menghasilkan *Skill Gap* beserta rekomendasi video pembelajarannya.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import json

## 1. Load Data
Memuat dataset sekunder (spesifikasi *skill* industri) dan dataset primer (*Skill Dictionary* berisi tautan YouTube). Kemudian, membuat kamus pemetaan dari nama *skill* ke *link* belajarnya untuk mempercepat pencarian (*lookup*).

In [2]:
# Load Datasets
df_jobs = pd.read_csv('IT_Job_Roles_Skills_Clean.csv')
df_dict = pd.read_csv('Skill_Dictionary.csv')

# Create mapping for skill to youtube link
skill_links = dict(zip(df_dict['Skills'].str.lower(), df_dict['Link Youtube']))

## 2. Model Training & Serialization
Melatih algoritma `TfidfVectorizer` dengan menggunakan keseluruhan kumpulan data *skill* industri untuk membangun korpus pembobotan vektor teks.
Setelah itu, hasil matriks komputasi setiap *role* beserta *vectorizer*-nya diekspor menggunakan `joblib` (`.pkl`) agar dapat dibaca (*load*) secara cepat pada arsitektur *Backend* tanpa perlu melatih ulang.

In [3]:
# Model Training: Fit TF-IDF Vectorizer on all job skills corpus
corpus = df_jobs['Skills'].tolist()
vectorizer = TfidfVectorizer()
vectorizer.fit(corpus)

# Transform all job skills into vectors
job_vectors = vectorizer.transform(corpus)

# Model Serialization (Save model for backend API usage)
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(job_vectors, 'job_vectors.pkl')

['job_vectors.pkl']

## 3. Recommendation Engine (Matching System)
Fungsi utama `recommend_learning_path` untuk menghitung jarak kesamaan profil pengguna dengan target pekerjaan. Fungsi ini juga menghitung **Skill Gap** (keterampilan yang disyaratkan industri namun belum dikuasai) dan memetakannya dengan tautan kurasi materi YouTube.

In [4]:
def recommend_learning_path(current_skills, target_role):
    current_skills = [skill.lower().strip() for skill in current_skills]
    target_role_lower = target_role.strip().lower()
    
    # Find the requested role
    role_idx = df_jobs.index[df_jobs['Job Title'].str.lower() == target_role_lower].tolist()
    if not role_idx:
        return {"Error": f"Role '{target_role}' not found in dataset."}
    
    idx = role_idx[0]
    role_data = df_jobs.iloc[idx]
    
    # Extract target skills for the role
    target_skills_str = role_data['Skills']
    target_skills = [skill.strip().lower() for skill in target_skills_str.split(',')]
    
    # Calculate Similarity Score
    user_skills_str = ", ".join(current_skills)
    user_vector = vectorizer.transform([user_skills_str])
    target_vector = job_vectors[idx]
    similarity_score = cosine_similarity(user_vector, target_vector)[0][0]
    
    # Skill Gap Analysis
    skill_gap = [skill for skill in target_skills if skill not in current_skills]
    
    # Map missing skills to YouTube links
    recommendations = []
    for skill in skill_gap:
        link = skill_links.get(skill, "Link not available")
        recommendations.append({"Skill": skill, "Link Youtube": link})
        
    return {
        "Target Role": role_data['Job Title'],
        "Similarity Score": f"{similarity_score * 100:.2f}%",
        "Missing Skills to Learn": recommendations
    }

## 4. Model Testing
Melakukan dua skenario pengujian dengan variasi input pengguna dan target pekerjaan untuk memastikan *matching engine* bekerja dengan baik.

In [5]:
# Test 1: Data Scientist
user_skills_1 = ["python", "sql", "data analysis"]
target_role_1 = "Data Scientist"

result_1 = recommend_learning_path(user_skills_1, target_role_1)
print("Test 1 Result:")
print(json.dumps(result_1, indent=4))

Test 1 Result:
{
    "Target Role": "Data Scientist",
    "Similarity Score": "41.91%",
    "Missing Skills to Learn": [
        {
            "Skill": "r",
            "Link Youtube": "https://youtube.com/playlist?list=PLIeJsyt_FUfJS6o2fMNuI7hGeBpqRkYMS&si=Ya01ntU2JTu5J0yH"
        },
        {
            "Skill": "machine learning",
            "Link Youtube": "https://youtube.com/playlist?list=PL2O3HdJI4voHNEv59SdXKRQVRZAFmwN9E&si=1sxoAw8wYw5xSqot"
        },
        {
            "Skill": "data science",
            "Link Youtube": "https://youtube.com/playlist?list=PLIeJsyt_FUfIeFxVGGNtXrdtaa83NVA2y&si=mINhQoF2mlOBbT-o"
        },
        {
            "Skill": "statistics",
            "Link Youtube": "https://youtube.com/playlist?list=PL0o_zxa4K1BVsziIRdfv4Hl4UIqDZhXWV&si=3Id1IwdsH3uNepn6"
        },
        {
            "Skill": "hadoop",
            "Link Youtube": "https://youtube.com/playlist?list=PLYwpaL_SFmcAhiP6C1qVorA7HZRejRE6M&si=76jBWIWSr8OxGPxZ"
        },
        {

In [6]:
# Test 2: DevOps Engineer
user_skills_2 = ["python", "linux", "networking"]
target_role_2 = "DevOps Engineer"

result_2 = recommend_learning_path(user_skills_2, target_role_2)
print("\nTest 2 Result:")
print(json.dumps(result_2, indent=4))


Test 2 Result:
{
    "Target Role": "DevOps Engineer",
    "Similarity Score": "38.08%",
    "Missing Skills to Learn": [
        {
            "Skill": "docker",
            "Link Youtube": "https://youtube.com/playlist?list=PL-CtdCApEFH-A7jBmdertzbeACuQWvQao&si=wQANrCJhwDdtQVU9"
        },
        {
            "Skill": "kubernetes",
            "Link Youtube": "https://youtube.com/playlist?list=PL-CtdCApEFH8XrWyQAyRd6d_CKwxD8Ime&si=vbpIAY8ZikImS9I2"
        },
        {
            "Skill": "cicd",
            "Link Youtube": "https://youtu.be/v-pqoP6ro-8?si=HnwGl0Mynh2jBSHW"
        },
        {
            "Skill": "aws",
            "Link Youtube": "https://youtube.com/playlist?list=PL2O3HdJI4voHl6yAoFI7UR4ne2CkWyVFB&si=yAQ1q3MEBCmzXCb8"
        },
        {
            "Skill": "azure",
            "Link Youtube": "https://youtube.com/playlist?list=PLvuXEs4kRMl_lCoL2FMZjaIFKYprROyUR&si=kggFhGjg4m4fBZoY"
        },
        {
            "Skill": "terraform",
            "Link Yo